In [1]:
# 0. Bibliotecas utilizadas
import os
import pandas as pd
import pyodbc
import warnings
from datetime import datetime

# 1. Tira mensagens de warning
warnings.filterwarnings('ignore')

# 2. Pega usuário de rede
usuario = os.getenv('USERNAME')

# 3. Caminho e carregamento do arquivo com credenciais (somente servidor e banco)
path = os.path.join(rf'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS')
arquivo_senhas = os.path.join(path, 'SENHAS.xlsx')

if not os.path.exists(arquivo_senhas):
    raise FileNotFoundError(f"❌ Arquivo de senhas não encontrado em: {arquivo_senhas}")

df_senhas = pd.read_excel(arquivo_senhas)
server, database = df_senhas.iloc[0, 0:2]  # Pega apenas server e database

# 4. Conexão com o SQL Server via autenticação integrada
try:
    print("🔌 Conectando ao banco de dados via autenticação integrada...")
    conn = pyodbc.connect(
        f'DRIVER={{ODBC Driver 17 for SQL Server}};'
        f'SERVER={server};DATABASE={database};Trusted_Connection=yes;'
    )
    print("✅ Conectado com sucesso.")
except Exception as e:
    raise ConnectionError(f"❌ Erro ao conectar com o banco de dados: {e}")

# 5. Query SQL com categorização por diferença de dias
query_table = """
SELECT 
    FT.AnoMesRef,
    FT.ID_Produto,
    FT.ID_Comissionado,
    FT.ID_UF,
    FT.ID_Cota,
    FT.ST_Contrato,
    FT.Tipo_Pessoa,
    FT.Grupo,
    FT.Cota,
    FT.Faixa_Atraso,
    FT.Desclassificado,
    FT.TP_Pagto_DemaisParcelas,
    FT.ST_Adimplencia,
    DP.CD_InscricaoNacional,
    VA.DT_EntregaBem,
    CO.CANAL_VENDA,
    VA.DT_Alocacao AS DT_Contemplacao,

    -- Categorização de tempo entre data de contemplação e entrega
    CASE 
        WHEN VA.DT_Alocacao IS NULL OR VA.DT_EntregaBem IS NULL THEN NULL
        WHEN DATEDIFF(DAY, VA.DT_Contemplacao, VA.DT_EntregaBem) <= 30 THEN 'Over 30'
        WHEN DATEDIFF(DAY, VA.DT_Alocacao, VA.DT_EntregaBem) <= 60 THEN 'Over 60'
        WHEN DATEDIFF(DAY, VA.DT_Alocacao, VA.DT_EntregaBem) <= 90 THEN 'Over 90'
        WHEN DATEDIFF(DAY, VA.DT_Alocacao, VA.DT_EntregaBem) <= 120 THEN 'Over 120'
        WHEN DATEDIFF(DAY, VA.DT_Alocacao, VA.DT_EntregaBem) <= 150 THEN 'Over 150'
        WHEN DATEDIFF(DAY, VA.DT_Alocacao, VA.DT_EntregaBem) <= 180 THEN 'Over 180'
        WHEN DATEDIFF(DAY, VA.DT_Alocacao, VA.DT_EntregaBem) <= 210 THEN 'Over 210'
        ELSE 'Over > 210'
    END AS Categoria_Tempo_Entrega

FROM 
    FT0015_CarteiraCotas AS FT
LEFT JOIN 
    DM0013_Pessoas AS DP ON FT.ID_Pessoa = DP.ID_Pessoa
LEFT JOIN 
    FT0002_VendaAlocacoes AS VA ON FT.ID_Cota = VA.ID_Cota
LEFT JOIN 
    DM0004_Comissionados AS CO ON FT.ID_Comissionado = CO.ID_Comissionado
WHERE
    FT.AnoMesRef >= '202401'
"""

# 6. Executa a consulta
print("📥 Executando a consulta...")
df = pd.read_sql(query_table, conn)

# 7. Converte colunas de data
df['DT_EntregaBem'] = pd.to_datetime(df['DT_EntregaBem'], errors='coerce')
df['DT_Contemplacao'] = pd.to_datetime(df['DT_Contemplacao'], errors='coerce')

# 8. Caminho para salvar o CSV
arquivo_saida_csv = os.path.join(
    rf'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos',
    'relatorio_contemplacao_entrega.csv'
)

# 9. Exporta como CSV
print("💾 Exportando para CSV...")
df.to_csv(arquivo_saida_csv, sep=';', index=False, encoding='utf-8-sig')

print(f"✅ Arquivo CSV exportado com sucesso para: {arquivo_saida_csv}")


🔌 Conectando ao banco de dados via autenticação integrada...


ConnectionError: ❌ Erro ao conectar com o banco de dados: ('HY000', '[HY000] [Microsoft][ODBC Driver 17 for SQL Server]SQL Server Network Interfaces: Credenciais não disponíveis no pacote de segurança\r\n (-2146893042) (SQLDriverConnect); [HY000] [Microsoft][ODBC Driver 17 for SQL Server]Cannot generate SSPI context (-2146893042); [HY000] [Microsoft][ODBC Driver 17 for SQL Server]SQL Server Network Interfaces: Credenciais não disponíveis no pacote de segurança\r\n (-2146893042); [HY000] [Microsoft][ODBC Driver 17 for SQL Server]Cannot generate SSPI context (-2146893042)')